# 📊 高级专家 笔试模拟练习四：SQL 编程实战专场

> **🎯 考核目标**：SQL 占目标大厂/目标大厂B笔试约 **40%** 权重。本练习使用 Python 内置的 `sqlite3` 模块在本地搭建迷你数据库，让你直接写真实 SQL。每道题都配有模拟数据、空白答题区和参考答案。

> **⚠️ 重要提示**：真实笔试写的是 Hive/Spark SQL，语法略有差异（如 `DATE_ADD`、`LATERAL VIEW`）。但核心逻辑一致，本练习侧重逻辑训练。

---

## 🛠️ 环境初始化（运行一次即可）

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# 创建内存数据库
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# 辅助函数：执行 SQL 并返回 DataFrame
def run_sql(sql, conn=conn):
    """执行 SQL 并以 DataFrame 格式展示结果"""
    try:
        return pd.read_sql_query(sql, conn)
    except Exception as e:
        print(f'❌ SQL 执行错误: {e}')
        return None

print('✅ 数据库环境已就绪！')

✅ 数据库环境已就绪！


## 📦 构建模拟数据表

In [2]:
np.random.seed(42)

# ============================================================
# 表1: user_actions (用户行为表) — 核心表
# ============================================================
n_actions = 5000
user_ids = np.random.randint(1001, 1201, n_actions)  # 200个用户
# 生成 30 天内的随机时间戳
base_date = datetime(2026, 2, 1)
action_times = [base_date + timedelta(
    days=np.random.randint(0, 30),
    hours=np.random.randint(0, 24),
    minutes=np.random.randint(0, 60)
) for _ in range(n_actions)]
event_types = np.random.choice(['browse', 'click', 'add_cart', 'purchase', 'refund'], 
                               n_actions, p=[0.4, 0.25, 0.15, 0.15, 0.05])

df_actions = pd.DataFrame({
    'user_id': user_ids,
    'event_time': [t.strftime('%Y-%m-%d %H:%M:%S') for t in action_times],
    'event_type': event_types,
    'event_date': [t.strftime('%Y-%m-%d') for t in action_times]
})
df_actions.to_sql('user_actions', conn, index=False, if_exists='replace')

# ============================================================
# 表2: orders (订单表)
# ============================================================
purchase_mask = df_actions['event_type'] == 'purchase'
n_orders = purchase_mask.sum()
df_orders = pd.DataFrame({
    'order_id': range(1, n_orders + 1),
    'user_id': df_actions.loc[purchase_mask, 'user_id'].values,
    'order_time': df_actions.loc[purchase_mask, 'event_time'].values,
    'order_date': df_actions.loc[purchase_mask, 'event_date'].values,
    'amount': np.round(np.random.lognormal(5, 1, n_orders), 2),  # 对数正态分布模拟订单金额
    'category': np.random.choice(['服装', '电子', '食品', '家居', '美妆'], n_orders)
})
df_orders.to_sql('orders', conn, index=False, if_exists='replace')

# ============================================================
# 表3: user_profile (用户画像表)
# ============================================================
all_users = sorted(df_actions['user_id'].unique())
df_profile = pd.DataFrame({
    'user_id': all_users,
    'gender': np.random.choice(['M', 'F'], len(all_users)),
    'age': np.random.randint(18, 60, len(all_users)),
    'city': np.random.choice(['上海', '杭州', '北京', '深圳', '广州'], len(all_users)),
    'register_date': [(base_date - timedelta(days=np.random.randint(30, 365))).strftime('%Y-%m-%d') for _ in all_users],
    'vip_level': np.random.choice([0, 1, 2, 3], len(all_users), p=[0.5, 0.25, 0.15, 0.1])
})
df_profile.to_sql('user_profile', conn, index=False, if_exists='replace')

# ============================================================
# 表4: ad_exposure (广告曝光表)
# ============================================================
n_ads = 2000
ad_times = [base_date + timedelta(
    days=np.random.randint(0, 30),
    hours=np.random.randint(0, 24)
) for _ in range(n_ads)]
df_ads = pd.DataFrame({
    'user_id': np.random.choice(all_users, n_ads),
    'ad_id': np.random.choice(['AD_A', 'AD_B', 'AD_C', 'AD_D', 'AD_E'], n_ads),
    'view_time': [t.strftime('%Y-%m-%d %H:%M:%S') for t in ad_times],
    'view_date': [t.strftime('%Y-%m-%d') for t in ad_times]
})
df_ads.to_sql('ad_exposure', conn, index=False, if_exists='replace')

# ============================================================
# 表5: experiment_log (实验分流表)
# ============================================================
exp_users = np.random.choice(all_users, 150, replace=False)
df_exp = pd.DataFrame({
    'user_id': exp_users,
    'experiment_group': np.random.choice(['control', 'treatment_A', 'treatment_B'], 150, p=[0.34, 0.33, 0.33]),
    'metric_value': np.round(np.random.normal(100, 25, 150), 2)
})
df_exp.to_sql('experiment_log', conn, index=False, if_exists='replace')

print(f'✅ 数据表构建完成！')
print(f'   user_actions: {len(df_actions)} 行')
print(f'   orders:       {len(df_orders)} 行')
print(f'   user_profile: {len(df_profile)} 行')
print(f'   ad_exposure:  {len(df_ads)} 行')
print(f'   experiment_log: {len(df_exp)} 行')

✅ 数据表构建完成！
   user_actions: 5000 行
   orders:       738 行
   user_profile: 200 行
   ad_exposure:  2000 行
   experiment_log: 150 行


---

## 🧪 第一题：窗口函数 — 用户首次购买日期 ⭐

**📝 题目**：
从 `user_actions` 表中，提取每个用户**第一次** `purchase` 行为的完整记录（user_id, event_time）。

**要求**：使用 `ROW_NUMBER() OVER()` 窗口函数实现，不要用 `GROUP BY + MIN` 后再 JOIN 的低效写法。

**考点**：窗口函数基础 — `ROW_NUMBER` + `PARTITION BY` + `ORDER BY`

In [13]:
# df_actions.info()
df_actions.head()
# df_actions['event_type'].unique()
df_actions[df_actions['user_id'] == 1103].sort_values(by='event_time') # 看看数据长什么样

,user_id,event_time,event_type,event_date
859,1103,2026-02-03 19:46:00,browse,2026-02-03
1645,1103,2026-02-04 16:04:00,purchase,2026-02-04
2699,1103,2026-02-05 01:41:00,add_cart,2026-02-05
3851,1103,2026-02-05 04:56:00,click,2026-02-05
1358,1103,2026-02-06 21:52:00,browse,2026-02-06
8,1103,2026-02-10 06:15:00,click,2026-02-10
1445,1103,2026-02-13 12:00:00,browse,2026-02-13
455,1103,2026-02-14 07:59:00,click,2026-02-14
2001,1103,2026-02-15 17:49:00,purchase,2026-02-15
2703,1103,2026-02-16 09:55:00,refund,2026-02-16


In [17]:
# ==================
# ✏️ 你的 SQL 写在这里：
# ==================
# 这次的新答案：
# 目标：提取用户首次购物行为的完整记录：browse > click_add_cart > pruchase > refund(可能有可能没有) > browse > ...
# 思路：基于purchase行为拆分用户购物记录:purchase1,purchase2,purchase3...然后截取purchase1的数据
# STEP 1 标记每一次用户的purchaserank
# STEP 2 根据purchase rank继续提取首次购买和下一次购买之间的记录
sql_q1 = """
WITH tmp AS(
    select a.*
        ,first_value(event_time) over (partition by user_id,event_type order by event_time) as first_event_time
        ,row_number() over (partition by user_id,event_type order by event_time) as user_event_rank
        ,row_number() over (partition by user_id order by event_time) as event_rank 
    from user_actions a 
),
purchase_tb AS (
    SELECT a.*
    FROM tmp a 
    WHERE event_type = 'purchase'
),
final_tb AS(
    SELECT a.* 
    FROM tmp a 
    JOIN (
        SELECT a.*
        FROM purchase_tb a 
        WHERE 1=1
        AND user_event_rank = 1 -- 保留首次purchase记录
    ) b on a.user_id = b.user_id and a.event_time <= b.first_event_time -- 原始记录表中同用户id记录数据发生在首次purchase时间前的所有记录
)

SELECT * FROM final_tb

/* 验证脚本：最终user_id数和原始有purchase行为用户id数一样（去重）
-- 执行结果通过 193 :193
SELECT COUNT(DISTINCT user_id) AS final_user_cnt, MAX(0) AS original_user_cnt
FROM final_tb 
UNION ALL 
SELECT MAX(0) AS final_user_cnt, COUNT(DISTINCT user_id) AS original_user_cnt
FROM final_tb 
WHERE 1=1
AND event_type = 'purchase'
*/
"""

run_sql(sql_q1)
# ==================

,user_id,event_time,event_type,event_date,first_event_time,user_event_rank,event_rank
0,1001,2026-02-03 00:46:00,browse,2026-02-03,2026-02-03 00:46:00,1,1
1,1001,2026-02-03 16:40:00,browse,2026-02-03,2026-02-03 00:46:00,2,2
2,1001,2026-02-03 21:10:00,browse,2026-02-03,2026-02-03 00:46:00,3,3
3,1001,2026-02-03 22:13:00,purchase,2026-02-03,2026-02-03 22:13:00,1,4
4,1002,2026-02-01 02:11:00,browse,2026-02-01,2026-02-01 02:11:00,1,1
...,...,...,...,...,...,...,...
1148,1200,2026-02-04 22:59:00,click,2026-02-04,2026-02-01 20:56:00,2,3
1149,1200,2026-02-05 10:39:00,click,2026-02-05,2026-02-01 20:56:00,3,4
1150,1200,2026-02-06 01:48:00,click,2026-02-06,2026-02-01 20:56:00,4,5
1151,1200,2026-02-06 10:18:00,browse,2026-02-06,2026-02-04 22:56:00,2,6


In [ ]:
# ==================
# ✏️ 你的 SQL 写在这里：
# ===之前的错误答案：没有注意到题目要求“提取每个用户**第一次** `purchase` 行为的完整记录”===

# sql_q1 = """
# WITH tmp AS(
#     select a.*,row_number() over (partition by user_id,event_type order by event_time) as user_event_rank
#     from user_actions a 
# )

# SELECT * from tmp where user_event_rank = 1 and event_type = 'purchase'
# """

# run_sql(sql_q1)
# ==================

,user_id,event_time,event_type,event_date,user_event_rank
0,1001,2026-02-03 22:13:00,purchase,2026-02-03,1
1,1002,2026-02-06 16:11:00,purchase,2026-02-06,1
2,1003,2026-02-15 17:42:00,purchase,2026-02-15,1
3,1004,2026-02-04 15:23:00,purchase,2026-02-04,1
4,1005,2026-02-04 11:32:00,purchase,2026-02-04,1
...,...,...,...,...,...
188,1196,2026-02-03 16:18:00,purchase,2026-02-03,1
189,1197,2026-02-16 18:20:00,purchase,2026-02-16,1
190,1198,2026-02-12 22:14:00,purchase,2026-02-12,1
191,1199,2026-02-07 12:36:00,purchase,2026-02-07,1


In [ ]:
# 💡 参考答案（做完再看！）
sql_q1_answer = """
WITH ranked AS (
    SELECT 
        user_id,
        event_time,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_time) AS rn
    FROM user_actions
    WHERE event_type = 'purchase'
)
SELECT user_id, event_time AS first_purchase_time
FROM ranked
WHERE rn = 1
ORDER BY user_id
LIMIT 10
"""
# run_sql(sql_q1_answer)  # 取消注释查看结果

---

## 🧪 第二题：多表 JOIN + 聚合 — 用户消费能力画像 ⭐

**📝 题目**：
关联 `orders` 和 `user_profile` 表，统计每个城市（city）中：
1. 总订单数
2. 总消费金额
3. 人均消费金额（总金额 / 去重用户数）
4. 按人均消费金额降序排列

**考点**：JOIN + GROUP BY + 多指标聚合 + 去重计数

In [26]:
# ==================
# ✏️ 你的 SQL 写在这里：
sql_q2 = """
WITH tmp AS(
    SELECT a.*
        ,coalesce(b.city,'unknown') as city 
        -- ,coalesce(b.gender,'unknown') as gender 
        -- ,coalesce(b.age,'unknown') as age 
        -- ,coalesce(b.job,'unknown') as job 
        -- ,coalesce(b.register_date,'unknown') as register_date 
        -- ,coalesce(b.vip_level,'unknown') as vip_level 
    FROM orders a 
    left join user_profile b 
    on a.user_id = b.user_id
),
city_tb AS (
    SELECT city
        ,COUNT(DISTINCT order_id) AS order_cnts
        ,SUM(amount) AS total_amount
        ,(SUM(amount) * 1.00) / (COUNT(DISTINCT order_id) * 1.00) AS avg_amount
    FROM tmp 
    GROUP BY city
)

SELECT *,row_number()over (order by avg_amount DESC) as city_avg_amount_rank
from city_tb a 
WHERE 1=1 
order by avg_amount desc
"""

run_sql(sql_q2)
# ==================

,city,order_cnts,total_amount,avg_amount,city_avg_amount_rank
0,广州,157,46744.00,297.732484,1
1,上海,158,44181.30,279.628481,2
2,深圳,144,39410.10,273.681250,3
3,北京,172,33907.06,197.134070,4
4,杭州,107,19379.48,181.116636,5


In [ ]:
# 💡 参考答案
sql_q2_answer = """
SELECT 
    p.city,
    COUNT(o.order_id) AS total_orders,
    ROUND(SUM(o.amount), 2) AS total_amount,
    ROUND(SUM(o.amount) / COUNT(DISTINCT o.user_id), 2) AS avg_amount_per_user
FROM orders o
JOIN user_profile p ON o.user_id = p.user_id
GROUP BY p.city
ORDER BY avg_amount_per_user DESC
"""
# run_sql(sql_q2_answer)

---

## 🧪 第三题：留存率计算 — 次日留存 & 7日留存 ⭐⭐

**📝 题目**：
基于 `user_actions` 表，计算 2026 年 2 月每天的**次日留存率**和 **7日留存率**。

定义：
- 某天的活跃用户 = 该天有任意行为记录的去重 user_id 集合
- 次日留存率 = 第 T 天活跃且第 T+1 天也活跃的用户数 / 第 T 天活跃用户数
- 7日留存率 = 第 T 天活跃且第 T+7 天也活跃的用户数 / 第 T 天活跃用户数

**考点**：自关联（SELF JOIN）+ 日期运算 + 去重 + 留存分析逻辑（高频必考！）

In [58]:
# ==================
# ✏️ 你的 SQL 写在这里：
sql_q3 = """
WITH user_tb AS (
    SELECT user_id
    FROM user_actions
    GROUP BY user_id
),
date_tb AS (
    SELECT event_date
    FROM user_actions
    GROUP BY event_date
),
full_tb AS (
    -- 构建日期✖️用户维度表
    SELECT a.*,b.user_id
    from date_tb a 
    cross join user_tb b 
),
daily_tb AS (
    select distinct a.*
        ,if(b.event_date = a.event_date,1,0) as is_active
    from full_tb a 
    left join (
        SELECT user_id,event_date
        FROM user_actions
        GROUP BY 1,2
    ) b on a.user_id = b.user_id and a.event_date = b.event_date
),
cal_tb as (
    SELECT a.*
        ,coalesce(lag(is_active,1) over (partition by user_id order by event_date),0) as is_active_1d
        ,coalesce(lag(is_active,2) over (partition by user_id order by event_date),0) as is_active_2d
        ,coalesce(lag(is_active,3) over (partition by user_id order by event_date),0) as is_active_3d
        ,coalesce(lag(is_active,4) over (partition by user_id order by event_date),0) as is_active_4d
        ,coalesce(lag(is_active,5) over (partition by user_id order by event_date),0) as is_active_5d
        ,coalesce(lag(is_active,6) over (partition by user_id order by event_date),0) as is_active_6d
        ,coalesce(lag(is_active,7) over (partition by user_id order by event_date),0) as is_active_7d
    FROM daily_tb a 
),
day_cnt AS (
    SELECT event_date
        ,COUNT(DISTINCT IF(is_active = 1,user_id,null)) as active_users_cnt
        ,COUNT(DISTINCT IF(is_active = 1 and is_active_1d = 1,user_id,null)) as active_1d_users_cnt
        ,COUNT(DISTINCT IF(is_active = 1 and user_daily_cnt =7,user_id,null)) as active_7d_users_cnt
    FROM (
        select a.*
            ,is_active_1d + is_active_2d + is_active_3d + is_active_4d + is_active_5d + is_active_6d + is_active_7d as user_daily_cnt
        FROM cal_tb a 
    ) a 
    GROUP BY 1 
)

SELECT event_date
    ,active_1d_users_cnt * 1.00 / active_users_cnt * 1.00 as rate_1d
    ,active_7d_users_cnt * 1.00 / active_users_cnt * 1.00 as rate_7d
FROM day_cnt
WHERE 1=1
"""

run_sql(sql_q3)
# ==================

,event_date,rate_1d,rate_7d
0,2026-02-01,0.000000,0.000000
1,2026-02-02,0.552381,0.000000
2,2026-02-03,0.552632,0.000000
3,2026-02-04,0.530973,0.000000
4,2026-02-05,0.540541,0.000000
5,2026-02-06,0.594059,0.000000
6,2026-02-07,0.517241,0.000000
7,2026-02-08,0.588785,0.000000
8,2026-02-09,0.571429,0.000000
9,2026-02-10,0.495327,0.009346


In [ ]:
# 💡 参考答案
sql_q3_answer = """
WITH daily_users AS (
    SELECT DISTINCT user_id, event_date
    FROM user_actions
)
SELECT 
    a.event_date,
    COUNT(DISTINCT a.user_id) AS dau,
    COUNT(DISTINCT b.user_id) AS retained_d1,
    ROUND(1.0 * COUNT(DISTINCT b.user_id) / COUNT(DISTINCT a.user_id), 4) AS retention_d1,
    COUNT(DISTINCT c.user_id) AS retained_d7,
    ROUND(1.0 * COUNT(DISTINCT c.user_id) / COUNT(DISTINCT a.user_id), 4) AS retention_d7
FROM daily_users a
LEFT JOIN daily_users b 
    ON a.user_id = b.user_id 
    AND b.event_date = DATE(a.event_date, '+1 day')
LEFT JOIN daily_users c 
    ON a.user_id = c.user_id 
    AND c.event_date = DATE(a.event_date, '+7 day')
GROUP BY a.event_date
ORDER BY a.event_date
LIMIT 15
"""
# run_sql(sql_q3_answer)

---

## 🧪 第四题：排名与 Top N — 各品类销量 Top 3 ⭐

**📝 题目**：
从 `orders` 表中，找出每个品类（category）消费金额最高的 Top 3 用户，输出：category, user_id, total_amount, ranking。

**考点**：窗口函数 `RANK()` 或 `DENSE_RANK()` — 分组内排名是面试最高频题型之一

In [61]:
# ==================
# ✏️ 你的 SQL 写在这里：
sql_q4 = """
WITH tmp as (
    SELECT category
        ,user_id
        ,sum(amount) as total_amount
    FROM orders 
    GROUP BY 1,2
)
SELECT * 
FROM (
SELECT a.*
    ,row_number() over (partition by category order by total_amount desc ) as ranking 
from tmp a 
)
WHERE 1=1
AND ranking <=3
order by category,total_amount desc 

"""

run_sql(sql_q4)
# ==================

,category,user_id,total_amount,ranking
0,家居,1194,4312.20,1
1,家居,1103,2330.35,2
2,家居,1004,1546.00,3
3,服装,1143,1901.55,1
4,服装,1032,1734.71,2
5,服装,1010,1308.98,3
6,电子,1019,1952.61,1
7,电子,1032,1423.39,2
8,电子,1099,1252.17,3
9,美妆,1111,2934.40,1


In [ ]:
# 💡 参考答案
sql_q4_answer = """
WITH user_category_total AS (
    SELECT 
        category,
        user_id,
        ROUND(SUM(amount), 2) AS total_amount
    FROM orders
    GROUP BY category, user_id
),
ranked AS (
    SELECT *,
        RANK() OVER (PARTITION BY category ORDER BY total_amount DESC) AS ranking
    FROM user_category_total
)
SELECT category, user_id, total_amount, ranking
FROM ranked
WHERE ranking <= 3
ORDER BY category, ranking
"""
# run_sql(sql_q4_answer)

---

## 🧪 第五题：转化漏斗分析 ⭐⭐

**📝 题目**：
基于 `user_actions` 表，计算用户行为漏斗转化率：

**browse → click → add_cart → purchase**

统计每个步骤的去重用户数和相对于上一步的转化率。

**考点**：CASE WHEN + 条件聚合 + 漏斗分析（业务分析高频题）

In [65]:
# ==================
# ✏️ 你的 SQL 写在这里：
sql_q5 = """
WITH tmp as (
    SELECT event_type
        ,COUNT(DISTINCT user_id) as user_cnt 
    from user_actions 
    GROUP BY 1 
)

SELECT SUM(IF(event_type = 'browse',user_cnt,0)) AS browse_user_cnt
    ,SUM(IF(event_type = 'click',user_cnt,0)) AS click_user_cnt
    ,SUM(IF(event_type = 'click',user_cnt,0)) * 1.00 / SUM(IF(event_type = 'browse',user_cnt,0)) * 1.00 as browse_to_click_rate
    ,SUM(IF(event_type = 'add_cart',user_cnt,0)) AS add_cart_user_cnt
    ,SUM(IF(event_type = 'add_cart',user_cnt,0)) * 1.00 / SUM(IF(event_type = 'click',user_cnt,0)) * 1.00 as click_to_add_cart_rate
    ,SUM(IF(event_type = 'purchase',user_cnt,0)) AS purchase_user_cnt
    ,SUM(IF(event_type = 'purchase',user_cnt,0)) * 1.00 / SUM(IF(event_type = 'add_cart',user_cnt,0)) * 1.00 as add_cart_to_purchase_rate
FROM tmp 

"""

run_sql(sql_q5)
# ==================

,browse_user_cnt,click_user_cnt,browse_to_click_rate,add_cart_user_cnt,click_to_add_cart_rate,purchase_user_cnt,add_cart_to_purchase_rate
0,200,200,1.0,197,0.985,193,0.979695


In [ ]:
# 💡 参考答案
sql_q5_answer = """
WITH funnel AS (
    SELECT
        COUNT(DISTINCT CASE WHEN event_type = 'browse' THEN user_id END) AS step1_browse,
        COUNT(DISTINCT CASE WHEN event_type = 'click' THEN user_id END) AS step2_click,
        COUNT(DISTINCT CASE WHEN event_type = 'add_cart' THEN user_id END) AS step3_cart,
        COUNT(DISTINCT CASE WHEN event_type = 'purchase' THEN user_id END) AS step4_purchase
    FROM user_actions
)
SELECT 
    step1_browse,
    step2_click,
    ROUND(1.0 * step2_click / step1_browse, 4) AS browse_to_click_rate,
    step3_cart,
    ROUND(1.0 * step3_cart / step2_click, 4) AS click_to_cart_rate,
    step4_purchase,
    ROUND(1.0 * step4_purchase / step3_cart, 4) AS cart_to_purchase_rate,
    ROUND(1.0 * step4_purchase / step1_browse, 4) AS overall_conversion
FROM funnel
"""
# run_sql(sql_q5_answer)

---

## 🧪 第六题：连续活跃天数 ⭐⭐⭐ (目标大厂高频难题)

**📝 题目**：
计算每个用户在 2 月份的**最大连续活跃天数**（活跃 = 当天有任意行为）。

**提示**：经典解法是利用 `日期 - ROW_NUMBER` 生成分组 key，相同 key 表示连续天。

**考点**：连续问题建模 — 这是目标大厂/目标大厂B笔试中最经典的 SQL 难题之一！

In [ ]:
# ==================
# ✏️ 你的 SQL 写在这里：
sql_q6 = """
WITH daily_users AS (
    SELECT DISTINCT user_id, event_date FROM user_actions
),
flagged AS (
    SELECT *,
        -- 和前一天比，不连续就标1（新段开始）
        CASE WHEN julianday(event_date) - julianday(
            LAG(event_date) OVER (PARTITION BY user_id ORDER BY event_date)
        ) = 1 THEN 0 ELSE 1 END AS new_group
    FROM daily_users
),
grouped AS (
    -- 累加标记 → 变成分组ID
    SELECT *,
        SUM(new_group) OVER (PARTITION BY user_id ORDER BY event_date) AS group_id
    FROM flagged
)
/*
SELECT user_id, MAX(streak) AS max_consecutive_days
FROM (
    SELECT user_id, group_id, COUNT(*) AS streak
    FROM grouped
    GROUP BY user_id, group_id
) t
GROUP BY user_id
*/

select * from grouped
"""

run_sql(sql_q6)

# ==================

,user_id,event_date,new_group,group_id
0,1001,2026-02-03,1,1
1,1001,2026-02-05,1,2
2,1001,2026-02-06,0,2
3,1001,2026-02-07,0,2
4,1001,2026-02-10,1,3
...,...,...,...,...
3335,1200,2026-02-23,1,4
3336,1200,2026-02-24,0,4
3337,1200,2026-02-26,1,5
3338,1200,2026-02-27,0,5


In [77]:
# 💡 参考答案
sql_q6_answer = """
WITH daily_active AS (
    -- 步骤1: 去重，获取每个用户每天的活跃记录
    SELECT DISTINCT user_id, event_date
    FROM user_actions
),
numbered AS (
    -- 步骤2: 为每个用户的活跃日编号
    SELECT 
        user_id,
        event_date,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_date) AS rn
    FROM daily_active
),
grouped AS (
    -- 步骤3: 日期减去行号 = 连续分组 key
    -- 如果连续活跃，date - rn 的值相同
    SELECT 
        user_id,
        event_date,
        rn,
        DATE(event_date, '-' || rn || ' day') AS group_key
    FROM numbered
),
streaks AS (
    -- 步骤4: 按分组 key 统计每段连续天数
    SELECT 
        user_id,
        group_key,
        COUNT(*) AS consecutive_days,
        MIN(event_date) AS start_date,
        MAX(event_date) AS end_date
    FROM grouped
    GROUP BY user_id, group_key
)
/*
-- 步骤5: 取每个用户最大连续天数
SELECT 
    user_id,
    MAX(consecutive_days) AS max_consecutive_days
FROM streaks
GROUP BY user_id
ORDER BY max_consecutive_days DESC
LIMIT 10
*/

select * from streaks
"""
run_sql(sql_q6_answer)

,user_id,group_key,consecutive_days,start_date,end_date
0,1001,2026-02-02,1,2026-02-03,2026-02-03
1,1001,2026-02-03,3,2026-02-05,2026-02-07
2,1001,2026-02-05,3,2026-02-10,2026-02-12
3,1001,2026-02-06,1,2026-02-14,2026-02-14
4,1001,2026-02-07,5,2026-02-16,2026-02-20
...,...,...,...,...,...
1516,1200,2026-02-02,8,2026-02-04,2026-02-11
1517,1200,2026-02-08,2,2026-02-18,2026-02-19
1518,1200,2026-02-11,2,2026-02-23,2026-02-24
1519,1200,2026-02-12,2,2026-02-26,2026-02-27


---

## 🧪 第七题：广告归因 — 最终点击归因模型 ⭐⭐⭐

**📝 题目**：
关联 `ad_exposure` 和 `orders` 表，实现**最终点击归因**：

1. 对每笔订单，找到该用户在下单**前 7 天内**的最后一次广告曝光
2. 将该广告视为归因广告
3. 统计每个 ad_id 的归因订单数和归因 GMV

**考点**：多表关联 + 时间窗口过滤 + 窗口函数排序取最近一条（广告归因是电商分析必考题）

In [90]:
# ==================
# ✏️ 你的 SQL 写在这里：
sql_q7 = """
with tmp as (
        SELECT a.*
        ,b.ad_id
        ,b.view_date
        ,row_number() over (partition by a.user_id order by b.view_date desc) as view_rank
    FROM orders a
    LEFT JOIN ad_exposure b 
    on a.user_id = b.user_id 
    and julianday(a.order_date) - julianday(b.view_date) <= 7
    and a.order_date >= b.view_date
)




SELECT ad_id
    ,COUNT(DISTINCT order_id) as order_cnt
    ,sum(amount) as order_amount
FROM tmp 
WHERE 1=1
AND view_rank = 1
GROUP BY 1
"""

run_sql(sql_q7)
# ==================

,ad_id,order_cnt,order_amount
0,NaN,5,1283.22
1,AD_A,29,6957.84
2,AD_B,41,12824.04
3,AD_C,50,11387.12
4,AD_D,39,10173.08
5,AD_E,29,10432.16


In [ ]:
# 💡 参考答案
sql_q7_answer = """
WITH order_ad_joined AS (
    -- 步骤1: 关联订单和广告曝光，限制曝光在下单前且7天内
    SELECT 
        o.order_id,
        o.user_id,
        o.order_time,
        o.amount,
        a.ad_id,
        a.view_time,
        -- 按曝光时间倒序排列，取最近一次
        ROW_NUMBER() OVER (
            PARTITION BY o.order_id 
            ORDER BY a.view_time DESC
        ) AS rn
    FROM orders o
    JOIN ad_exposure a 
        ON o.user_id = a.user_id
        AND a.view_time < o.order_time                        -- 曝光在下单之前
        AND a.view_time >= DATETIME(o.order_time, '-7 day')   -- 且在7天窗口内
),
last_touch AS (
    -- 步骤2: 只保留最近一次曝光（最终点击归因）
    SELECT * FROM order_ad_joined WHERE rn = 1
)
-- 步骤3: 按广告 ID 汇总归因效果
SELECT 
    ad_id,
    COUNT(DISTINCT order_id) AS attributed_orders,
    ROUND(SUM(amount), 2) AS attributed_gmv
FROM last_touch
GROUP BY ad_id
ORDER BY attributed_gmv DESC
"""
# run_sql(sql_q7_answer)

---

## 🧪 第八题：A/B 实验结果分析 ⭐⭐

**📝 题目**：
基于 `experiment_log` 表和 `orders` 表：

1. 关联实验分组信息，计算各组（control / treatment_A / treatment_B）的**人均订单金额**和**购买转化率**（有订单的用户 / 该组总用户）
2. 排除异常用户：订单数 > 10 的用户视为异常，需剔除

**考点**：LEFT JOIN + 条件过滤异常 + HAVING + 分组聚合（实验分析核心SQL）

In [100]:
# ==================
# ✏️ 你的 SQL 写在这里：
sql_q8 = """
with tmp as (
    SELECT experiment_group
        ,a.user_id
        ,count(distinct a.user_id) as user_cnt
        ,count(distinct b.order_id) as order_cnt
        ,SUM(metric_value) as metric_value
    FROM experiment_log a 
    left join orders b on a.user_id = b.user_id
    GROUP BY 1,2
)

SELECT experiment_group
    ,sum(user_cnt) as user_cnt
    ,sum(if(order_cnt >0,user_cnt,0)) as order_user_cnt
    ,sum(order_cnt) as order_cnt
    ,sum(metric_value) as metric_value
    ,sum(metric_value)/sum(user_cnt) as metric_value_per_user
    ,sum(if(order_cnt >0,user_cnt,0)) *1.00 / sum(user_cnt) as order_rate
FROM tmp
WHERE 1=1
AND order_cnt <= 10 
GROUP BY 1

"""

run_sql(sql_q8)
# ==================

,experiment_group,user_cnt,order_user_cnt,order_cnt,metric_value,metric_value_per_user,order_rate
0,control,49,46,177,17815.31,363.577755,0.938776
1,treatment_A,53,52,204,19965.52,376.707925,0.981132
2,treatment_B,48,47,173,16516.18,344.087083,0.979167


In [ ]:
# 💡 参考答案
sql_q8_answer = """
WITH valid_users AS (
    -- 步骤1: 排除异常用户（订单数 > 10）
    SELECT user_id
    FROM orders
    GROUP BY user_id
    HAVING COUNT(*) <= 10
),
exp_orders AS (
    -- 步骤2: 关联实验分组与订单（仅保留合法用户）
    SELECT 
        e.experiment_group,
        e.user_id,
        COALESCE(SUM(o.amount), 0) AS total_amount,
        COUNT(o.order_id) AS order_count
    FROM experiment_log e
    LEFT JOIN orders o ON e.user_id = o.user_id
    WHERE e.user_id IN (SELECT user_id FROM valid_users)
       OR e.user_id NOT IN (SELECT DISTINCT user_id FROM orders)
    GROUP BY e.experiment_group, e.user_id
)
-- 步骤3: 按实验组汇总
SELECT 
    experiment_group,
    COUNT(*) AS total_users,
    SUM(CASE WHEN order_count > 0 THEN 1 ELSE 0 END) AS buyers,
    ROUND(1.0 * SUM(CASE WHEN order_count > 0 THEN 1 ELSE 0 END) / COUNT(*), 4) AS conversion_rate,
    ROUND(SUM(total_amount) / COUNT(*), 2) AS avg_amount_per_user
FROM exp_orders
GROUP BY experiment_group
ORDER BY experiment_group
"""
# run_sql(sql_q8_answer)

---

## 📌 考点覆盖对照表

| 题号 | 考点 | 难度 | 目标大厂考频 |
| :---: | :--- | :---: | :---: |
| Q1 | 窗口函数 ROW_NUMBER | ⭐ | 🔥🔥🔥 必考 |
| Q2 | 多表 JOIN + 聚合 | ⭐ | 🔥🔥🔥 必考 |
| Q3 | 留存率计算（自关联） | ⭐⭐ | 🔥🔥🔥 必考 |
| Q4 | 分组内 Top N (RANK) | ⭐ | 🔥🔥 高频 |
| Q5 | 转化漏斗 (CASE WHEN) | ⭐⭐ | 🔥🔥🔥 必考 |
| Q6 | 连续活跃天数 | ⭐⭐⭐ | 🔥🔥🔥 经典难题 |
| Q7 | 广告归因（时间窗口） | ⭐⭐⭐ | 🔥🔥 电商高频 |
| Q8 | 实验结果 SQL 分析 | ⭐⭐ | 🔥🔥 高频 |